# Ранжирование русскоязычных поисковых результатов — финальное решение

Полный пайплайн финального сабмита (Public LB **0.960916**): препроцессинг → обучение трёх реранкеров → смесь → `outputs/submission.csv`.

| Модель | Дообучение | NDCG@10, фолд 0 |
|---|---|---|
| `BAAI/bge-reranker-v2-m3` (cross-encoder) | полное, 5 фолдов, 1 эпоха | 0.9342 |
| `Qwen/Qwen3-Reranker-4B` | LoRA, фолды 1–4, 1 эпоха, вход 448 токенов | 0.9542 |
| `Qwen/Qwen3-Reranker-8B` | LoRA, фолды 1–4, половина эпохи, вход 1024 токена | 0.9581 |
| **Смесь z-оценок: `qwen8 + qwen4 + 0.5 · ce`** | | **0.9650** |

**Система.** Обучение и инференс: **Linux**, GPU NVIDIA Tesla T4 16 ГБ (Kaggle); 8B — на двух T4.

**Требования к железу.** CE и Qwen 4B — одна GPU с 16 ГБ; Qwen 8B — 2×16 ГБ или одна GPU от 32 ГБ (модели сами распределяются по всем видимым GPU).
Время на T4: CE ~4 ч, Qwen 4B ~8 ч, Qwen 8B ~7.5 ч. Файлы соревнования (`train.csv`, `test.csv`, `sample_submission.csv`) — в `DATA_DIR`.
Обучение на GPU недетерминировано, поэтому повторный запуск даёт близкие, но не побитово равные скоры.

## 0. Параметры

In [ ]:
DATA_DIR = '.'       # папка с train.csv, test.csv, sample_submission.csv
OUT_DIR = 'outputs'  # предсказания моделей, адаптеры LoRA и submission.csv
WEIGHTS = {'qwen8': 1.0, 'qwen4': 1.0, 'ce': 0.5}  # веса z-оценок в смеси (подобраны на фолде 0)
SEED = 42
VALID_FOLD = 0  # Qwen обучаются на остальных фолдах, фолд 0 — проверка смеси

CE = {
    'model': 'BAAI/bge-reranker-v2-m3',
    'epochs': 1, 'lr': 1e-5, 'max_len': 384, 'accum_queries': 4, 'batch_size': 64,
}
QWEN = {
    'qwen4': {'model': 'Qwen/Qwen3-Reranker-4B', 'revision': '22e683669bc0f0bd69640a1354a6d0aebcfeede5',
              'train_max_len': 448, 'train_fraction': 1.0, 'train_token_budget': 8192},
    'qwen8': {'model': 'Qwen/Qwen3-Reranker-8B', 'revision': '77d193c791ed757ca307ee72715aa132723da912',
              'train_max_len': 1024, 'train_fraction': 0.5, 'train_token_budget': 4096},
}
LORA = {'r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'bias': 'none',
        'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']}
QWEN_LR = 1e-4
QWEN_ACCUM_QUERIES = 4
QWEN_MAX_CANDIDATES = 12   # при обучении: все релевантные + случайные нерелевантные
QWEN_MAX_LEN = 1024        # вход при предсказании (запрос + документ, без служебной части промпта)
QWEN_TOKEN_BUDGET = 8192   # токенов в батче предсказаний
INSTRUCTION = 'Given a question in Russian, retrieve Wikipedia passages that answer the question'

In [ ]:
import gc
import html
import math
import os
import random
import re
import time
import unicodedata
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import numpy as np
import pandas as pd
import torch
import transformers
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

DATA_DIR, OUT_DIR = Path(DATA_DIR), Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
autocast = lambda: torch.autocast(device_type='cuda', dtype=torch.float16)
print('GPU:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], '| torch', torch.__version__,
      '| transformers', transformers.__version__)

## 1. Препроцессинг

Найдено в EDA: кракозябры (UTF-8, прочитанный как cp1251), HTML-теги и многократно экранированные сущности, вики-разметка, знаки ударения
(в 9% текстов и 23% запросов), разложенные «й»/«ё», невидимые символы и спецпробелы; в заголовках — заглушки и хвост « — Википедия».
Очистка их убирает, пустые заголовки восстанавливаются по `article_id` (у каждой статьи после очистки ровно один заголовок).
Модели получают пару «запрос» — «заголовок. фрагмент».

In [ ]:
ACCENTS = '[\N{COMBINING GRAVE ACCENT}\N{COMBINING ACUTE ACCENT}]'
INVISIBLE = '[\N{SOFT HYPHEN}\N{ZERO WIDTH SPACE}\N{ZERO WIDTH NON-JOINER}\N{ZERO WIDTH JOINER}\N{ZERO WIDTH NO-BREAK SPACE}]'
TITLE_SUFFIX = r'\s+[-—]\s+Википедия$'
TITLE_PLACEHOLDERS = {'', '—', '-', 'Не указано'}


def fix_mojibake(s):
    """UTF-8, ошибочно прочитанный как cp1251. Для нормального русского текста decode падает — строка не меняется."""
    try:
        return s.encode('cp1251').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s


def unescape_html(s):
    """Раскрывает HTML-сущности, в том числе многократно экранированные (&amp;lt;p&amp;gt; → &lt;p&gt; → <p>)."""
    while (unescaped := html.unescape(s)) != s:
        s = unescaped
    return s


def strip_accents(s):
    """Удаляет ударения и собирает разложенные символы (и + ˘ → й, е + ¨ → ё)."""
    s = unicodedata.normalize('NFD', s)
    s = re.sub(ACCENTS, '', s)
    return unicodedata.normalize('NFC', s)


def clean_text(s):
    if pd.isna(s):
        return s
    s = fix_mojibake(s)
    s = unescape_html(s)
    s = re.sub(r'<[a-zA-Z/][^>]*>', ' ', s)                   # HTML-теги
    s = re.sub(r'\[\[(?:[^\]|]*\|)?([^\]]*)\]\]', r'\1', s)  # [[ссылка|текст]] → текст
    s = re.sub(r'\{\{[^}]*\}\}', ' ', s)                       # {{шаблоны}}
    s = strip_accents(s)
    s = re.sub(INVISIBLE, '', s)
    s = re.sub(r'\s*\(\s*[,;]?\s*\)', '', s)                  # пустые скобки на месте удалённых транскрипций
    s = re.sub(r'\s+', ' ', s)                               # неразрывные и узкие пробелы, переводы строк → пробел
    s = re.sub(r' ([,.;:])', r'\1', s)                        # пробел перед знаком препинания
    return s.strip()


def clean_title(s):
    s = clean_text(s)
    if pd.isna(s):
        return np.nan
    s = re.sub(TITLE_SUFFIX, '', s)
    return np.nan if s in TITLE_PLACEHOLDERS else s


train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
full = pd.concat([train.assign(part='train'), test.assign(part='test')], ignore_index=True)
full['query'] = full['query'].map(clean_text)
full['text'] = full['text'].map(clean_text)
full['title'] = full['title'].map(clean_title)
title_by_article = full.dropna(subset=['title']).groupby('article_id')['title'].agg(lambda s: s.mode()[0])
full['title'] = full['title'].fillna(full['article_id'].map(title_by_article))
full['doc'] = np.where(full['title'].notna(), full['title'] + '. ' + full['text'], full['text'])

COLUMNS = ['id', 'query_id', 'query', 'doc']
train = full.loc[full['part'].eq('train'), COLUMNS + ['relevance']].reset_index(drop=True)
test = full.loc[full['part'].eq('test'), COLUMNS].reset_index(drop=True)
print('train:', train.shape, train['query_id'].nunique(), 'запросов | test:', test.shape, test['query_id'].nunique(), 'запросов')

## 2. Валидация и метрика

5 фолдов GroupKFold по запросам (все кандидаты запроса в одном фолде). Метрика — средний по запросам NDCG@10 по правилам соревнования:
при равных скорах кандидаты упорядочиваются по `id` как строке.

In [ ]:
queries = train.drop_duplicates('query_id')[['query_id']].reset_index(drop=True)
queries['fold'] = -1
for fold, (_, valid_idx) in enumerate(GroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(queries, groups=queries['query_id'])):
    queries.loc[valid_idx, 'fold'] = fold
train['fold'] = train['query_id'].map(queries.set_index('query_id')['fold'])
valid = train[train['fold'] == VALID_FOLD]


def dcg_by_query(df, k):
    rank = df.groupby('query_id').cumcount() + 1
    gain = (2 ** df['relevance'] - 1) / np.log2(rank + 1)
    return gain.where(rank <= k, 0).groupby(df['query_id']).sum()


def ndcg10(df, scores, k=10):
    """Средний по запросам NDCG@k; df: query_id, id, relevance; scores — в порядке строк df."""
    df = df[['query_id', 'id', 'relevance']].assign(score=np.asarray(scores))
    ranked = df.sort_values(['query_id', 'score', 'id'], ascending=[True, False, True])
    ideal = df.sort_values(['query_id', 'relevance'], ascending=[True, False])
    return (dcg_by_query(ranked, k) / dcg_by_query(ideal, k).replace(0, np.nan)).fillna(0).mean()


def zscore(df, column):
    groups = df.groupby('query_id')[column]
    return ((df[column] - groups.transform('mean')) / groups.transform('std').replace(0, 1)).fillna(0)


print(queries.groupby('fold').size().to_dict(), '| фолд 0:', valid.shape)

## 3. Общее для моделей

Каждая модель даёт скоры для фолда 0 (проверка) и для теста; они сохраняются в `OUT_DIR`.

In [ ]:
def run_stage(name, compute):
    valid_scores, test_scores = compute()
    valid_scores.to_parquet(OUT_DIR / f'{name}_fold0.parquet', index=False)
    test_scores.to_parquet(OUT_DIR / f'{name}_test.parquet', index=False)
    assert np.isfinite(valid_scores[name]).all() and np.isfinite(test_scores[name]).all()
    print(f'{name}: NDCG@10 на фолде 0 = {ndcg10(valid, valid_scores[name].values):.4f}')
    return valid_scores, test_scores


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def listwise_loss(scores, labels):
    """Softmax по кандидатам запроса: максимизируем вероятность релевантных среди соседей."""
    return -(torch.log_softmax(scores, dim=0) * labels).sum() / labels.sum()


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

## 4. Cross-encoder `bge-reranker-v2-m3`

Полное дообучение на каждом из 5 фолдов (обучение на остальных четырёх), 1 эпоха. Один шаг обучения — запрос со всеми кандидатами,
listwise-лосс (softmax по кандидатам запроса), шаг оптимизатора раз в 4 запроса, fp16 + GradScaler, gradient checkpointing.
Для теста скоры пяти моделей усредняются, для проверки смеси берётся модель, не видевшая фолд 0.

In [ ]:
def ce_predict(model, tokenizer, df):
    model.eval()
    order = np.argsort(df['doc'].str.len().values)
    queries, docs = df['query'].values[order], df['doc'].values[order]
    scores = np.empty(len(df), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(df), CE['batch_size']):
            batch = tokenizer(list(queries[start:start + CE['batch_size']]), list(docs[start:start + CE['batch_size']]),
                              truncation='only_second', max_length=CE['max_len'], padding=True, return_tensors='pt').to('cuda')
            with autocast():
                scores[order[start:start + CE['batch_size']]] = model(**batch).logits[:, 0].float().cpu().numpy()
    return scores


def ce_train_fold(fold):
    seed_everything(SEED + fold)
    fit = train[train['fold'] != fold]
    tokenizer = AutoTokenizer.from_pretrained(CE['model'])
    model = AutoModelForSequenceClassification.from_pretrained(CE['model'], num_labels=1)
    model.gradient_checkpointing_enable()
    model = model.to('cuda')

    groups = [(g['query'].iat[0], g['doc'].tolist(), g['relevance'].to_numpy(np.float32)) for _, g in fit.groupby('query_id', sort=False)]
    steps = math.ceil(len(groups) / CE['accum_queries']) * CE['epochs']
    optimizer = torch.optim.AdamW(model.parameters(), lr=CE['lr'], weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler('cuda')
    for epoch in range(CE['epochs']):
        model.train()
        random.shuffle(groups)
        for i, (query, docs, labels) in enumerate(tqdm(groups, desc=f'CE, фолд {fold}', mininterval=60)):
            batch = tokenizer([query] * len(docs), docs, truncation='only_second', max_length=CE['max_len'], padding=True, return_tensors='pt').to('cuda')
            with autocast():
                scores = model(**batch).logits[:, 0].float()
            loss = listwise_loss(scores, torch.from_numpy(labels).to('cuda')) / CE['accum_queries']
            scaler.scale(loss).backward()
            if (i + 1) % CE['accum_queries'] == 0 or i + 1 == len(groups):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()

    valid_scores = ce_predict(model, tokenizer, valid) if fold == VALID_FOLD else None
    test_scores = ce_predict(model, tokenizer, test)
    del model, optimizer
    free_gpu()
    return valid_scores, test_scores


def ce_stage():
    test_scores = []
    for fold in range(5):
        fold_valid, fold_test = ce_train_fold(fold)
        test_scores.append(fold_test)
        if fold == VALID_FOLD:
            valid_scores = fold_valid
    return valid[['id']].assign(ce=valid_scores), test[['id']].assign(ce=np.mean(test_scores, axis=0))


ce_valid, ce_test = run_stage('ce', ce_stage)

## 5. Qwen3-Reranker + LoRA

Промпт и скор — как в карточке модели: инструкция, запрос и документ в чат-шаблоне, скор — логит «yes» минус логит «no» на последнем токене.
Инструкция уточнена для русского языка.

Обучение: LoRA r16 на всех линейных слоях, веса модели заморожены в fp16, адаптеры в fp32, autocast fp16 (на T4 нет bf16).
Последний слой декодера считается целиком в fp32: за ним стоит RMSNorm, которой безразличен масштаб входа, и LoRA раздувает его выход
выше предела fp16. Gradient checkpointing; лосс и шаг оптимизатора — как у CE, lr 1e-4, линейное расписание с прогревом 10%.
Кандидаты длинного запроса прогоняются кусками по `train_token_budget` токенов — лосс остаётся одним softmax по всем кандидатам.
Модель распределяется по всем видимым GPU (`device_map='auto'`).

Предсказание — на входе до 1024 токенов (обрезано ~1% пар, при 448 — 16%), батчи по бюджету токенов.

In [ ]:
QWEN_PREFIX = ('<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. '
               'Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')
QWEN_SUFFIX = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


class QwenReranker:
    def __init__(self, cfg):
        self.tokenizer = AutoTokenizer.from_pretrained(cfg['model'], revision=cfg['revision'], padding_side='left')
        self.prefix = self.tokenizer.encode(QWEN_PREFIX, add_special_tokens=False)
        self.suffix = self.tokenizer.encode(QWEN_SUFFIX, add_special_tokens=False)
        self.token_ids = [self.tokenizer.convert_tokens_to_ids('no'), self.tokenizer.convert_tokens_to_ids('yes')]
        seed_everything(SEED)
        max_memory = {i: int(torch.cuda.get_device_properties(i).total_memory * 0.85) for i in range(torch.cuda.device_count())}
        model = AutoModelForCausalLM.from_pretrained(cfg['model'], revision=cfg['revision'], dtype=torch.float16,
                                                     device_map='auto', max_memory=max_memory)
        model.config.use_cache = False
        run_in_fp32(model.model.layers[-1])
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        self.model = get_peft_model(model, LoraConfig(**LORA))
        assert all(p.dtype == torch.float32 for p in self.model.parameters() if p.requires_grad)

    def encode(self, queries, docs, max_len):
        texts = [f'<Instruct>: {INSTRUCTION}\n<Query>: {q}\n<Document>: {d}' for q, d in zip(queries, docs)]
        encoded = self.tokenizer(texts, padding=False, truncation='longest_first', return_attention_mask=False, max_length=max_len)
        return [self.prefix + ids + self.suffix for ids in encoded['input_ids']]

    def score(self, inputs):
        """Логит «yes» минус логит «no» на последнем токене."""
        batch = self.tokenizer.pad({'input_ids': inputs}, padding=True, return_tensors='pt').to('cuda:0')
        logits = self.model(**batch, logits_to_keep=1).logits[:, -1, self.token_ids].float()
        return logits[:, 1] - logits[:, 0]

    @torch.no_grad()
    def predict(self, df):
        self.model.eval()
        inputs = self.encode(df['query'].values, df['doc'].values, QWEN_MAX_LEN)
        scores = np.empty(len(df), dtype=np.float32)
        for batch in tqdm(batches_by_budget([len(ids) for ids in inputs], 16, QWEN_TOKEN_BUDGET), mininterval=60):
            with autocast():
                scores[batch] = self.score([inputs[i] for i in batch]).cpu().numpy()
        return scores

    def train(self, fit, cfg):
        groups = []
        for _, g in fit.groupby('query_id', sort=False):
            g = g.sample(frac=1, random_state=SEED).sort_values('relevance', ascending=False, kind='stable').head(QWEN_MAX_CANDIDATES)
            groups.append((g['query'].iat[0], g['doc'].tolist(), g['relevance'].to_numpy(np.float32)))
        random.Random(SEED).shuffle(groups)
        groups = groups[:int(len(groups) * cfg['train_fraction'])]

        trainable = [p for p in self.model.parameters() if p.requires_grad]
        steps = math.ceil(len(groups) / QWEN_ACCUM_QUERIES)
        optimizer = torch.optim.AdamW(trainable, lr=QWEN_LR, weight_decay=0.01)
        scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * steps), steps)
        scaler = torch.amp.GradScaler('cuda', init_scale=2 ** 12)  # при 2**16 градиенты fp16 переполняются
        start = time.time()
        self.model.train()
        for i, (query, docs, labels) in enumerate(tqdm(groups, desc=cfg['model'], mininterval=120)):
            inputs = self.encode([query] * len(docs), docs, cfg['train_max_len'])
            chunks = batches_by_budget([len(ids) for ids in inputs], len(inputs), cfg['train_token_budget'])
            with autocast():
                chunk_scores = torch.cat([self.score([inputs[j] for j in chunk]) for chunk in chunks])
            order = torch.from_numpy(np.concatenate(chunks)).to(chunk_scores.device)
            scores = torch.empty_like(chunk_scores).scatter(0, order, chunk_scores)
            loss = listwise_loss(scores, torch.from_numpy(labels).to(scores.device)) / QWEN_ACCUM_QUERIES
            scaler.scale(loss).backward()
            if (i + 1) % QWEN_ACCUM_QUERIES == 0 or i + 1 == len(groups):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
        print(f'обучение: {(time.time() - start) / 3600:.2f} ч, {len(groups)} запросов')


def batches_by_budget(lengths, max_size, budget):
    """Индексы по возрастанию длины, нарезанные на батчи: не больше max_size текстов и budget токенов с учётом дополнения."""
    batches, batch = [], []
    for i in np.argsort(lengths, kind='stable'):
        if batch and (len(batch) == max_size or (len(batch) + 1) * lengths[i] > budget):
            batches.append(batch)
            batch = []
        batch.append(i)
    return batches + [batch]


def to_fp32(value):
    if torch.is_tensor(value) and value.is_floating_point():
        return value.float()
    return tuple(to_fp32(v) for v in value) if isinstance(value, tuple) else value


def run_in_fp32(layer):
    """Слой считается в fp32 без autocast."""
    layer.float()
    forward = layer.forward

    def fp32_forward(hidden_states, *args, **kwargs):
        with torch.autocast(device_type='cuda', enabled=False):
            return forward(hidden_states.float(), *[to_fp32(a) for a in args], **{k: to_fp32(v) for k, v in kwargs.items()})

    layer.forward = fp32_forward


def qwen_stage(name):
    reranker = QwenReranker(QWEN[name])
    reranker.train(train[train['fold'] != VALID_FOLD], QWEN[name])
    reranker.model.save_pretrained(OUT_DIR / f'{name}_adapter')
    result = valid[['id']].assign(**{name: reranker.predict(valid)}), test[['id']].assign(**{name: reranker.predict(test)})
    del reranker
    free_gpu()
    return result

### 5.1. Qwen3-Reranker-4B: 1 эпоха на фолдах 1–4, обучение на 448 токенах, предсказания на 1024

In [ ]:
qwen4_valid, qwen4_test = run_stage('qwen4', lambda: qwen_stage('qwen4'))

### 5.2. Qwen3-Reranker-8B: половина эпохи на фолдах 1–4, обучение и предсказания на 1024 токенах

In [ ]:
qwen8_valid, qwen8_test = run_stage('qwen8', lambda: qwen_stage('qwen8'))

## 6. Смесь и сабмит

Скоры каждой модели нормируются внутри запроса (z-оценки) и складываются с весами `WEIGHTS`. На фолде 0 соседние веса
(qwen4 от 0.5 до 1, ce от 0.25 до 0.75) дают 0.9639–0.9650 — выбраны самые простые.

In [ ]:
def blend(df, parts):
    for name, scores in parts.items():
        df = df.merge(scores, on='id')
        df[name] = zscore(df, name)
    return df.assign(prediction=sum(w * df[name] for name, w in WEIGHTS.items()))


valid_blend = blend(valid[['id', 'query_id', 'relevance']], {'ce': ce_valid, 'qwen4': qwen4_valid, 'qwen8': qwen8_valid})
test_blend = blend(test[['id', 'query_id']], {'ce': ce_test, 'qwen4': qwen4_test, 'qwen8': qwen8_test})
print(pd.Series({name: ndcg10(valid_blend, valid_blend[name]) for name in WEIGHTS} | {'смесь': ndcg10(valid_blend, valid_blend['prediction'])})
      .rename('NDCG@10, фолд 0').round(4).to_string())

sample = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype={'id': str})
submission = sample[['id']].merge(test_blend[['id', 'prediction']].astype({'id': str}), on='id', how='left')
assert submission['prediction'].notna().all() and len(submission) == len(sample)
assert not test_blend.duplicated(['query_id', 'prediction']).any()
submission.to_csv(OUT_DIR / 'submission.csv', index=False)
print('сохранено:', OUT_DIR / 'submission.csv', submission.shape)